In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# Part 1: Testing Model Accuracy 
# ==========================================
print("⏳ Calculating model accuracy...")
# 1. Load the data that contains the actual targets
train_df = pd.read_csv('Data/train_scaled.csv')

# 2. Prepare the target (Poverty vs Non-Poverty) just like the model training
train_df['Target_Binary'] = train_df['Target'].replace({1: 1, 2: 1, 3: 1, 4: 0})
X = train_df.drop(columns=['Target', 'Target_Binary'])
y = train_df['Target_Binary']

# 3. Split 20% for testing (Unseen data for the model)
_, X_test_acc, _, y_test_acc = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Load the model and calculate accuracy
model = joblib.load('xgboost_poverty_model.pkl')
y_pred_acc = model.predict(X_test_acc)

accuracy = accuracy_score(y_test_acc, y_pred_acc) * 100
print(f"\n🎯 Model Accuracy on Test Data: {accuracy:.2f}%")
print("-" * 50)


# ==========================================
# Part 2: Generating and Saving Predictions for test.csv
# ==========================================
print("⏳ Generating predictions for test.csv and saving...")
# 1. Load the required datasets
test_df = pd.read_csv('Data/test.csv')
train_eng_df = pd.read_csv('Data/train_engineered.csv') 

# 2. Prepare the test data (Apply Feature Engineering)
X_test_eng = test_df.copy()
mapping = {'yes': 1, 'no': 0}
for col in ['dependency', 'edjefe', 'edjefa']:
    if col in X_test_eng.columns:
        X_test_eng[col] = X_test_eng[col].replace(mapping).astype(float)

# Apply customized ratios
X_test_eng['children_ratio'] = X_test_eng['r4t1'] / (X_test_eng['tamhog'] + 1e-5)
X_test_eng['rooms_per_person'] = X_test_eng['rooms'] / (X_test_eng['tamhog'] + 1e-5)

# 3. Align columns and apply Scaling
X_train_for_scale = train_eng_df.drop(columns=['Target', 'Target_Binary'], errors='ignore')
# Ensure test columns match train columns perfectly
X_test_aligned = X_test_eng.reindex(columns=X_train_for_scale.columns, fill_value=0)

scaler = MinMaxScaler()
scaler.fit(X_train_for_scale) 
X_test_scaled = scaler.transform(X_test_aligned) 

# 4. Extract predictions for the test dataset
test_predictions = model.predict(X_test_scaled)

# 5. Save the complete data along with predictions in a new file
test_df['AI_Prediction'] = test_predictions
test_df['Status'] = test_df['AI_Prediction'].map({1: 'Poverty', 0: 'Non-Poverty'})

# Save the file locally
output_filename = 'test_data_with_predictions.csv'
test_df.to_csv(output_filename, index=False)

print(f"✅ Data along with AI predictions successfully saved to: '{output_filename}'")

⏳ Calculating model accuracy...

🎯 Model Accuracy on Test Data: 75.97%
--------------------------------------------------
⏳ Generating predictions for test.csv and saving...
✅ Data along with AI predictions successfully saved to: 'test_data_with_predictions.csv'
